In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="distilbert_mean_cosine_median_threshold_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(model_name)
print("hidden_size:", model.config.hidden_size)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
hidden_size: 768


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    pooled = summed / counts
    pooled = torch.nn.functional.normalize(pooled, p=2, dim=-1)
    return pooled

def encode_sentences(texts, batch_size=64, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i + batch_size]
            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = model(**enc)
            pooled = mean_pool(outputs.last_hidden_state, enc["attention_mask"])
            all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0)


In [5]:
emb1 = encode_sentences(sent1, batch_size=64, max_length=128)
emb2 = encode_sentences(sent2, batch_size=64, max_length=128)

cosine_similarities = (emb1 * emb2).sum(dim=1).numpy()
threshold = float(np.median(cosine_similarities))
y_pred = (cosine_similarities >= threshold).astype(int)

print("emb1 shape:", tuple(emb1.shape))
print("emb2 shape:", tuple(emb2.shape))
print("similarity_stats:", {
    "min": float(cosine_similarities.min()),
    "median": float(np.median(cosine_similarities)),
    "max": float(cosine_similarities.max()),
})
print("threshold:", threshold)
print("done")


  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

emb1 shape: (408, 768)
emb2 shape: (408, 768)
similarity_stats: {'min': 0.7408234477043152, 'median': 0.9500881433486938, 'max': 0.9976521134376526}
threshold: 0.9500881433486938
done


In [ ]:

vault.create_record_list("threshold_cosine_mrpc_similarity_prediction", column_names=["prediction", "cosine_scores"])

for i in range(len(y_pred)):
    vault.append_record("threshold_cosine_mrpc_similarity_prediction", 
                        {
                            "prediction": y_pred[i],
                            "cosine_scores": float(cosine_similarities[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT threshold_cosine_mrpc_similarity_prediction"
embedding = get_embeddings(description)
vault.create_description("threshold_cosine_mrpc_similarity_prediction", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("threshold_cosine_mrpc_similarity_prediction", cat, embedding, prop)

In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
label_names = ["not_paraphrase", "paraphrase"]
report = classification_report(y_true, y_pred, target_names=label_names)
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=label_names))

for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(cosine_similarities[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", label_names[int(y_pred[i])])

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(cosine_similarities[i]), "threshold:", threshold)
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


{'accuracy': 0.6495098039215687, 'f1': 0.7039337474120083}
                precision    recall  f1-score   support

not_paraphrase       0.47      0.74      0.57       129
    paraphrase       0.83      0.61      0.70       279

      accuracy                           0.65       408
     macro avg       0.65      0.67      0.64       408
  weighted avg       0.72      0.65      0.66       408

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score: 0.9466805458068848
true: 1 pred: 0 label: not_paraphrase
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.893559455871582
true: 0 pred: 0 label: not_paraphrase
sentence1: The dollar was at

In [7]:
vault.create_record_list("distilbert_mean_cosine_median_threshold_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("distilbert_mean_cosine_median_threshold_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "threshold_cosine_mrpc_similarity_prediction": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT distilbert_mean_cosine_median_threshold_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("distilbert_mean_cosine_median_threshold_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_mean_cosine_median_threshold_mrpc_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'distilbert-base-uncased',
 'device': 'mps',
 'pooling': 'mean',
 'threshold_strategy': 'median_similarity',
 'threshold': 0.9500881433486938,
 'num_examples': 408,
 'accuracy': 0.6495098039215687,
 'f1': 0.7039337474120083}

In [ ]:
description = "INSERT TEXT HERE ABOUT distilbert_mean_cosine_median_threshold_mrpc process/notebook" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("distilbert_mean_cosine_median_threshold_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_mean_cosine_median_threshold_mrpc", cat, embedding, prop)